In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, ConfusionMatrixDisplay)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)


# Load your data
df = pd.read_csv("oil_presence_trainset.csv")

print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['Oil_Presence'].value_counts()}")
print(f"Class distribution (%):\n{df['Oil_Presence'].value_counts(normalize=True)}")


# Create a copy to avoid modifying original
data = df.copy()

# Replace NaN in categorical columns with 'None'
data['Trap_Type'] = data['Trap_Type'].fillna('None')
data['Rock_Type'] = data['Rock_Type'].fillna('None')

# Fill numerical NaN with mean (or use KNN like you did)
numeric_cols = ['Porosity', 'Permeability', 'Seismic_Score', 
                'Proximity_to_Oil_Field', 'Estimated_Reservoir_Depth']

# Simple imputation with mean
for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

print("Missing values after cleaning:")
print(data.isnull().sum())


# Separate features and target
X = data.drop('Oil_Presence', axis=1)
y = data['Oil_Presence']

# Identify categorical columns
categorical_cols = ['Rock_Type', 'Trap_Type']

# Method 1: Label Encoding (Simple)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"{col} encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Method 2: One-Hot Encoding (Alternative - use if you want more features)
# X = pd.get_dummies(X, columns=categorical_cols, dtype=float)
# print(f"Shape after one-hot encoding: {X.shape}")

# Scale numerical features (optional but recommended)
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print(f"\nFinal feature shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")




# Split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:\n{pd.Series(y_train).value_counts(normalize=True)}")
print(f"\nTest class distribution:\n{pd.Series(y_test).value_counts(normalize=True)}")



# Initialize Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf.fit(X_train, y_train)

# Make predictions
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

# Evaluate
print("=== BASIC RANDOM FOREST RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Oil', 'Oil Present']))



# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

# Grid Search
print("Starting Grid Search...")
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

# Get the best model
rf_best = grid_search.best_estimator_




# Make predictions with tuned model
y_pred_best = rf_best.predict(X_test)
y_pred_proba_best = rf_best.predict_proba(X_test)[:, 1]

print("=== TUNED RANDOM FOREST RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_best):.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba_best):.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No Oil', 'Oil Present']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(cm)




# Perform 5-fold cross-validation
cv_scores = cross_val_score(rf_best, X, y, cv=5, scoring='roc_auc')

print("\n=== CROSS-VALIDATION RESULTS ===")
print(f"Cross-validation AUC scores: {cv_scores}")
print(f"Mean AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Cross-validation with different metrics
cv_accuracy = cross_val_score(rf_best, X, y, cv=5, scoring='accuracy')
cv_precision = cross_val_score(rf_best, X, y, cv=5, scoring='precision')
cv_recall = cross_val_score(rf_best, X, y, cv=5, scoring='recall')
cv_f1 = cross_val_score(rf_best, X, y, cv=5, scoring='f1')

print(f"Mean Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
print(f"Mean Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})")
print(f"Mean Recall: {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
print(f"Mean F1-Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")



# Perform 5-fold cross-validation
cv_scores = cross_val_score(rf_best, X, y, cv=5, scoring='roc_auc')

print("\n=== CROSS-VALIDATION RESULTS ===")
print(f"Cross-validation AUC scores: {cv_scores}")
print(f"Mean AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Cross-validation with different metrics
cv_accuracy = cross_val_score(rf_best, X, y, cv=5, scoring='accuracy')
cv_precision = cross_val_score(rf_best, X, y, cv=5, scoring='precision')
cv_recall = cross_val_score(rf_best, X, y, cv=5, scoring='recall')
cv_f1 = cross_val_score(rf_best, X, y, cv=5, scoring='f1')

print(f"Mean Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
print(f"Mean Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})")
print(f"Mean Recall: {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
print(f"Mean F1-Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")



# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_best.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'][:10], feature_importance['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Most Important Features for Oil Presence Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



import joblib

# Save model and preprocessing objects
joblib.dump(rf_best, 'random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

print("Model and preprocessing objects saved successfully!")



def predict_oil_presence(new_data, model=rf_best, scaler=scaler, encoders=label_encoders):
    """
    Make predictions on new oil exploration data
    """
    # Create copy
    X_new = new_data.copy()
    
    # Encode categorical variables
    for col, le in encoders.items():
        if col in X_new.columns:
            X_new[col] = X_new[col].fillna('None')
            # Handle unknown categories
            X_new[col] = X_new[col].apply(lambda x: x if x in le.classes_ else 'None')
            X_new[col] = le.transform(X_new[col])
    
    # Scale numerical features
    numeric_cols = ['Porosity', 'Permeability', 'Seismic_Score', 
                    'Proximity_to_Oil_Field', 'Estimated_Reservoir_Depth']
    X_new[numeric_cols] = scaler.transform(X_new[numeric_cols])
    
    # Predict
    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)[:, 1]
    
    return predictions, probabilities

# Example usage
sample_data = pd.DataFrame({
    'Rock_Type': ['Sandstone'],
    'Porosity': [18.5],
    'Permeability': [650.0],
    'Trap_Type': ['Anticline'],
    'Seismic_Score': [0.75],
    'Proximity_to_Oil_Field': [3.2],
    'Estimated_Reservoir_Depth': [2800.0]
})

pred, prob = predict_oil_presence(sample_data)
print(f"\nSample Prediction:")
print(f"Prediction: {'Oil Present' if pred[0] == 1 else 'No Oil'}")
print(f"Probability: {prob[0]:.4f}")








def run_random_forest_pipeline(data_path):
    """
    Complete Random Forest pipeline for oil presence prediction
    """
    # Load data
    df = pd.read_csv(data_path)
    
    # Clean data
    data = df.copy()
    data['Trap_Type'] = data['Trap_Type'].fillna('None')
    data['Rock_Type'] = data['Rock_Type'].fillna('None')
    
    # Fill numerical NaN with mean
    numeric_cols = ['Porosity', 'Permeability', 'Seismic_Score', 
                    'Proximity_to_Oil_Field', 'Estimated_Reservoir_Depth']
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].mean())
    
    # Prepare features and target
    X = data.drop('Oil_Presence', axis=1)
    y = data['Oil_Presence']
    
    # Encode categorical variables
    categorical_cols = ['Rock_Type', 'Trap_Type']
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
    
    # Scale features
    scaler = StandardScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Train Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    
    # Evaluate
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    
    results = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_proba),
        'model': rf,
        'scaler': scaler,
        'encoders': label_encoders,
        'feature_importance': pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
    }
    
    return results

# Run the pipeline
results = run_random_forest_pipeline("oil_presence_trainset.csv")

print("\n=== FINAL RESULTS ===")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall: {results['recall']:.4f}")
print(f"F1-Score: {results['f1']:.4f}")
print(f"AUC-ROC: {results['auc']:.4f}")

print("\nTop 5 Important Features:")
print(results['feature_importance'].head(5))